# Prepare the Year 2 London survey file

This notebook prepares the final analysis file from the Year 2 SPSS dataset. It does four things:

1. Reads the list of 125 activity composites that are consistent across all eight survey years.
2. Keeps the requested geography, demographic, activity, volunteering, weight and survey fields.
3. Keeps respondents from 32 London local authorities, using `LA_2023` and excluding the City of London.
4. Writes a data CSV and a separate variable dictionary CSV.

The original SAV file is only used as an input. This notebook does not create or modify any SAV files. The output columns follow their order in the original SPSS file, apart from `year`, which is added as the second column.

In [1]:
from pathlib import Path
import json
import re

import pandas as pd
import pyreadstat

# Input files
data_path = Path('spss/spss28/active_lives_survey_nov_16-17_data_year_2_shared_20250106.sav')
activity_folder = Path(r'A:\eik\xwechat_files\wxid_5z3utjrxmj5322_82b8\msg\file\2026-07')
activity_files = []

# Identify the supplied workbook by its 125 unique entries, not by its filename.
for candidate in activity_folder.glob('*composites*.xlsx'):
    try:
        first_column = pd.read_excel(
            candidate, sheet_name=0, header=3, usecols=[0]
        ).iloc[:, 0].dropna()
    except (OSError, ValueError):
        continue

    if len(first_column) == 125 and first_column.nunique() == 125:
        activity_files.append(candidate)

if len(activity_files) != 1:
    raise FileNotFoundError(
        f'Expected one composites workbook in {activity_folder}, found {len(activity_files)}'
    )

activity_list_path = activity_files[0]

# Output files
data_output_path = Path('active_lives_1617_london_125.csv')
dictionary_output_path = Path('active_lives_1617_london_125_variables.csv')

survey_year = 2017
chunk_size = 20_000

if not data_path.exists():
    raise FileNotFoundError(f'SPSS input file not found: {data_path.resolve()}')

if not activity_list_path.exists():
    raise FileNotFoundError(f'Activity workbook not found: {activity_list_path}')

print(f'SPSS input: {data_path.resolve()}')
print(f'Activity workbook: {activity_list_path}')

SPSS input: Y:\afinal\UKDA-8391-spss\spss\spss28\active_lives_survey_nov_16-17_data_year_2_shared_20250106.sav
Activity workbook: A:\eik\xwechat_files\wxid_5z3utjrxmj5322_82b8\msg\file\2026-07\八年composites对比.xlsx


## 1. Read the activity list

The first worksheet contains the 125 composites whose definitions are unchanged across the eight survey years. The fourth row is the table header and the first column contains the variable suffixes.

In [2]:
activity_table = pd.read_excel(activity_list_path, sheet_name=0, header=3)

activity_suffixes = (
    activity_table.iloc[:, 0]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

if len(activity_suffixes) != 125:
    raise ValueError(f'Expected 125 activity suffixes, found {len(activity_suffixes)}')

if len(activity_suffixes) != len(set(activity_suffixes)):
    raise ValueError('The activity list contains duplicate suffixes')

activity_preview = activity_table.iloc[:, :4].head().copy()
activity_preview.columns = ['DV suffix', 'Activity name', 'Item count', 'Definition']

print(f'Activity suffixes read: {len(activity_suffixes)}')
display(activity_preview)

Activity suffixes read: 125


,DV suffix,Activity name,Item count,Definition
0,ABSEILING_H03,Abseiling,1,A1_3_4
1,ACTTRAV_C03,Active Travel,2,"CYC1_1, WALK1_1"
2,AIKIDO_S04,Aikido,1,A1_5_6_4
3,AIRGUN_S08,Airgun (including pistol),1,A1_5_7_1
4,ARCHERY_J01,Archery,2,"A1_4_2, A1_5_1"


## 2. Build the list of columns to keep

SPSS variable names are matched without regard to case. An activity variable is retained when its original SPSS name contains one of the requested activity prefixes and one of the 125 stable activity suffixes. Two additional summary variables are also retained by exact name: `MEMS7_ALL` and `MEMS7GR_ALL`.

The suffix checks are not case-sensitive, and the prefix and suffix can appear anywhere in the variable name. Other variables containing `ALL` are not retained unless they also match one of the 125 activity suffixes.


In [3]:
# Metadata can be read without loading the survey records.
_, metadata = pyreadstat.read_sav(data_path, metadataonly=True)
source_columns = metadata.column_names
source_column_set = set(source_columns)

geography_columns = ['LA_2023', 'Reg9', 'LondInOut']
demographic_columns = (
    ['Age16plus', 'Age9', 'Disab3']
    + [f'disty{i}_POP' for i in range(1, 14)]
)
volunteering_columns = (
    ['VOLANY']
    + [f'volint{i}' for i in range(1, 8)]
    + ['VOLFRQ_POP']
)
weight_columns = [name for name in source_columns if name.lower().startswith('wt_')]
other_columns = ['mode', 'serial', 'Number_Activities_150', 'month']

activity_prefixes = [
    'MEMS7_',
    'MEMS7GR_',
    'INOUTA_',
    'INOUTB_',
    'DAYS10P60GR_',
    'MONTHS_12_',
]

activity_columns = []
activity_check = []
activity_suffixes_lower = [suffix.lower() for suffix in activity_suffixes]
special_all_variables = {'mems7_all', 'mems7gr_all'}

for prefix in activity_prefixes:
    prefix_lower = prefix.lower()

    matched_by_suffix = {
        name
        for name in source_columns
        if prefix_lower in name.lower()
        and any(
            suffix_lower in name.lower()
            for suffix_lower in activity_suffixes_lower
        )
    }
    matched_special_all = {
        name
        for name in source_columns
        if name.lower() in special_all_variables
        and prefix_lower in name.lower()
    }

    # Reapply the source order after combining suffix matches with the two summaries.
    matched_variables = [
        name
        for name in source_columns
        if name in matched_by_suffix or name in matched_special_all
    ]
    matched_suffixes = {
        suffix
        for suffix in activity_suffixes
        if any(suffix.lower() in name.lower() for name in matched_variables)
    }

    activity_columns.extend(matched_variables)
    activity_check.append({
        'prefix': prefix,
        'activity_suffixes': len(activity_suffixes),
        'variables_found': len(matched_variables),
        'special_ALL': len(matched_special_all),
        'suffixes_found': len(matched_suffixes),
        'suffixes_missing': len(activity_suffixes) - len(matched_suffixes),
    })

# A source variable should only appear once even if a name meets more than one rule.
activity_columns = list(dict.fromkeys(activity_columns))

requested_columns = (
    geography_columns
    + demographic_columns
    + activity_columns
    + volunteering_columns
    + weight_columns
    + other_columns
)

missing_required_columns = [
    name for name in requested_columns if name not in source_column_set
]
if missing_required_columns:
    raise ValueError(f'Required columns not found: {missing_required_columns}')

if len(requested_columns) != len(set(requested_columns)):
    raise ValueError('The selected column list contains duplicates')

# Put the selected variables back into their original SPSS order.
requested_column_set = set(requested_columns)
source_ordered_columns = [
    name for name in source_columns if name in requested_column_set
]

display(pd.DataFrame(activity_check))
print(f'Activity variables selected: {len(activity_columns)}')
print(f'Source columns selected: {len(source_ordered_columns)}')


,prefix,activity_suffixes,variables_found,special_ALL,suffixes_found,suffixes_missing
0,MEMS7_,125,130,1,125,0
1,MEMS7GR_,125,131,1,125,0
2,INOUTA_,125,105,0,105,20
3,INOUTB_,125,105,0,105,20
4,DAYS10P60GR_,125,125,0,125,0
5,MONTHS_12_,125,125,0,125,0


Activity variables selected: 721
Source columns selected: 761


## 3. Identify the 32 retained London local authorities

The London filter is based directly on the value labels attached to `LA_2023`. London local authority codes run from `E09000001` to `E09000033`. `E09000001 City of London` is removed explicitly, leaving 32 local authorities in the analysis file.


In [4]:
la_label_name = metadata.variable_to_label['LA_2023']
la_value_labels = metadata.value_labels[la_label_name]

london_local_authorities = {
    value: label
    for value, label in la_value_labels.items()
    if re.match(r'^E090000\d{2}\b', label)
    and not label.startswith('E09000001 ')
}

if len(london_local_authorities) != 32:
    raise ValueError(
        f'Expected 32 London local authorities after excluding City of London, found {len(london_local_authorities)}'
    )

london_la_values = set(london_local_authorities)
print(f'London local authorities retained: {len(london_la_values)}')
display(
    pd.DataFrame(
        london_local_authorities.items(),
        columns=['LA_2023 value', 'LA_2023 label'],
    )
)

London local authorities retained: 32


,LA_2023 value,LA_2023 label
0,8.0,E09000002 Barking and Dagenham
1,9.0,E09000003 Barnet
2,17.0,E09000004 Bexley
3,30.0,E09000005 Brent
4,35.0,E09000006 Bromley
5,44.0,E09000007 Camden
6,68.0,E09000008 Croydon
7,78.0,E09000009 Ealing
8,91.0,E09000010 Enfield
9,107.0,E09000011 Greenwich


## 4. Read and filter the survey data

The file is read in chunks so that the full England dataset does not need to be held in memory. Each chunk is reduced to London records before the chunks are combined.

In [5]:
london_chunks = []
rows_read = 0

for chunk, _ in pyreadstat.read_file_in_chunks(
    pyreadstat.read_sav,
    data_path,
    chunksize=chunk_size,
    usecols=source_ordered_columns,
):
    rows_read += len(chunk)
    london_rows = chunk['LA_2023'].isin(london_la_values)
    london_chunks.append(chunk.loc[london_rows].copy())

london_data = pd.concat(london_chunks, ignore_index=True)
london_data = london_data.loc[:, source_ordered_columns]
london_data.insert(1, 'year', survey_year)

print(f'Rows read from SPSS: {rows_read:,}')
print(f'London respondents retained: {len(london_data):,}')
print(f'Output columns: {len(london_data.columns):,}')
display(london_data.head())

Rows read from SPSS: 196,635
London respondents retained: 19,248
Output columns: 762


,serial,year,mode,month,Age16plus,wt_final,wt_final_online,wt_final_B,wt_final_C,wt_final_AB,...,INOUTA_BOWLSCROWNGREEN_U19,INOUTB_BOWLSCROWNGREEN_U19,INOUTA_BOWLSFLATGREEN_U20,INOUTB_BOWLSFLATGREEN_U20,INOUTA_BOWLSSHORTMAT_U23,INOUTB_BOWLSSHORTMAT_U23,INOUTA_GYMNASTICSONLY_U24,INOUTB_GYMNASTICSONLY_U24,INOUTA_TRAMPOLINING_U25,INOUTB_TRAMPOLINING_U25
0,1.602901e+14,2017,1.0,22.0,1.0,0.689125,0.444363,NaN,0.531912,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.604901e+14,2017,1.0,15.0,1.0,4.817157,2.324213,NaN,2.416555,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.605900e+14,2017,1.0,20.0,1.0,0.331489,0.265711,0.231759,NaN,0.480566,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.605900e+14,2017,1.0,16.0,1.0,1.201918,0.833470,0.966493,NaN,2.055275,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.605900e+14,2017,1.0,16.0,1.0,0.674352,0.673429,0.718970,NaN,1.114718,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Create the variable dictionary

The dictionary contains one row for every output variable. Labels, formats, measurement levels and value labels come from the SPSS metadata. The added `year` variable is documented separately as a derived field.

In [6]:
label_by_name = dict(zip(metadata.column_names, metadata.column_labels))
source_position = {name: i + 1 for i, name in enumerate(source_columns)}
source_format = getattr(metadata, 'original_variable_types', {})
measurement_level = getattr(metadata, 'variable_measure', {})

geography_set = set(geography_columns)
demographic_set = set(demographic_columns)
volunteering_set = set(volunteering_columns)
weight_set = set(weight_columns)
other_set = set(other_columns)

def section_for(name):
    if name == 'year':
        return 'Survey information'
    if name in geography_set:
        return 'Geography'
    if name in demographic_set:
        return 'Demographics'
    if name in volunteering_set:
        return 'Volunteering'
    if name in weight_set:
        return 'Weights'
    if name in other_set:
        return 'Survey information'
    for prefix in activity_prefixes:
        if name.startswith(prefix):
            return f'Activity: {prefix.rstrip("_")}'
    return ''

def value_labels_for(name):
    label_name = metadata.variable_to_label.get(name)
    labels = metadata.value_labels.get(label_name, {})
    return json.dumps(labels, ensure_ascii=False) if labels else ''

dictionary_rows = []
for output_number, name in enumerate(london_data.columns, start=1):
    if name == 'year':
        dictionary_rows.append({
            'output_position': output_number,
            'variable_name': name,
            'section': section_for(name),
            'variable_label': 'Survey year',
            'source_position': '',
            'source_format': 'integer',
            'measurement_level': 'nominal',
            'value_labels': '',
            'notes': 'Derived from the Year 2 source file; constant value 2017',
        })
        continue

    dictionary_rows.append({
        'output_position': output_number,
        'variable_name': name,
        'section': section_for(name),
        'variable_label': label_by_name.get(name, ''),
        'source_position': source_position.get(name, ''),
        'source_format': source_format.get(name, ''),
        'measurement_level': measurement_level.get(name, ''),
        'value_labels': value_labels_for(name),
        'notes': '',
    })

variable_dictionary = pd.DataFrame(dictionary_rows)
display(variable_dictionary.head(10))

,output_position,variable_name,section,variable_label,source_position,source_format,measurement_level,value_labels,notes
0,1,serial,Survey information,Serial,1,F15.0,nominal,"{""-99.0"": ""Missing, should have been answered""...",
1,2,year,Survey information,Survey year,,integer,nominal,,Derived from the Year 2 source file; constant ...
2,3,mode,Survey information,Mode of completion,3,F8.2,nominal,"{""-99.0"": ""Missing, should have been answered""...",
3,4,month,Survey information,Interview month,6,F8.2,nominal,"{""-99.0"": ""Missing, should have been answered""...",
4,5,Age16plus,Demographics,Aged 16 and over,12,F8.2,nominal,"{""0.0"": ""Age under 16"", ""1.0"": ""All adults (ag...",
5,6,wt_final,Weights,All: Online and Postal,14,F8.2,scale,,
6,7,wt_final_online,Weights,All: Online,15,F8.2,scale,,
7,8,wt_final_B,Weights,Phase 2: Online Group 1,16,F8.2,scale,,
8,9,wt_final_C,Weights,Phase 2: Online Group 2,17,F8.2,scale,,
9,10,wt_final_AB,Weights,Phase 2: Postal & Online Group 1,18,F8.2,scale,,


## 6. Validate and save the two CSV files

The checks below confirm that all retained records are in the 32-authority London list excluding the City of London, `year` is the second column and has the value 2017, and the variable dictionary matches the data columns exactly. Both files use UTF-8 with a byte-order mark so that they open cleanly in Excel.

In [7]:
if not london_data['LA_2023'].isin(london_la_values).all():
    raise ValueError('The output contains records outside the London LA list')

if not london_data['Reg9'].eq(3).all():
    raise ValueError('The output contains a record whose Reg9 value is not London')

if not london_data['LondInOut'].isin([1, 2]).all():
    raise ValueError('The output contains a record outside Inner or Outer London')

if london_data.columns[1] != 'year':
    raise ValueError('year is not the second output column')

if not london_data['year'].eq(2017).all():
    raise ValueError('year contains a value other than 2017')

if variable_dictionary['variable_name'].tolist() != london_data.columns.tolist():
    raise ValueError('The variable dictionary does not match the data columns')

london_data.to_csv(
    data_output_path,
    index=False,
    encoding='utf-8-sig',
    float_format='%.15g',
)
variable_dictionary.to_csv(
    dictionary_output_path,
    index=False,
    encoding='utf-8-sig',
)

print(f'Data file: {data_output_path.resolve()}')
print(f'Data shape: {london_data.shape}')
print(f'Variable dictionary: {dictionary_output_path.resolve()}')
print(f'Dictionary rows: {len(variable_dictionary):,}')

Data file: Y:\afinal\UKDA-8391-spss\active_lives_1617_london_125.csv
Data shape: (19248, 762)
Variable dictionary: Y:\afinal\UKDA-8391-spss\active_lives_1617_london_125_variables.csv
Dictionary rows: 762
